# PUDM Extension — Point Cloud Upsampling

This notebook runs **PUDM** (Point Cloud Upsampling via Denoising Diffusion Model) with swappable generative strategies:
- **DDPM** (baseline, from CVPR 2024 paper)
- **Flow Matching** (new alternative)

**Requirements:** Colab GPU runtime (T4 or better).

## 1. Setup & Installation

In [ ]:
import os
import subprocess

# Clone the repo from GitHub
REPO_DIR = '/content/pudm_extension'
GITHUB_URL = 'https://github.com/valbad/pudm_extension.git'

if not os.path.exists(REPO_DIR):
    result = subprocess.run(['git', 'clone', GITHUB_URL, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f'git clone failed:\n{result.stderr}\n'
            'Make sure the repo is PUBLIC at https://github.com/valbad/pudm_extension/settings'
        )
    print(f'Cloned repo to {REPO_DIR}')
else:
    print(f'Repo already exists at {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Mount Google Drive (for data & checkpoints only)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install Python dependencies
!pip install -q torch numpy h5py einops open3d transforms3d Pillow tensorboard termcolor tqdm ninja

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Cache compiled CUDA extensions on Google Drive to avoid recompilation
import shutil, sys, glob as _glob, time as _time

CACHE_BASE = '/content/drive/MyDrive/MVA/pointcloud/cache'
# Version cache by Python + PyTorch + CUDA to avoid binary incompatibilities
_cache_tag = f'py{sys.version_info.major}{sys.version_info.minor}_pt{torch.__version__.split("+")[0]}_cu{torch.version.cuda.replace(".", "")}'
POINTOPS_CACHE = os.path.join(CACHE_BASE, 'pointops', _cache_tag)
PNET2_CACHE    = os.path.join(CACHE_BASE, 'pointnet2_ops', _cache_tag)

os.makedirs(CACHE_BASE, exist_ok=True)

# --- Compile pointops (or restore from cache) ---
os.chdir(REPO_DIR)
POINTOPS_SRC = os.path.join(REPO_DIR, 'src', 'ops', 'pointops')

cached_sos = _glob.glob(os.path.join(POINTOPS_CACHE, 'pointops_cuda*.so'))
if cached_sos:
    for so in cached_sos:
        shutil.copy2(so, POINTOPS_SRC)
    print(f'✓ pointops: restored .so from Drive cache')
else:
    print('pointops: building from scratch (first time for this env)...')
    os.environ.pop('TORCH_CUDA_ARCH_LIST', None)
    %cd {POINTOPS_SRC}
    !python setup.py build_ext --inplace 2>&1 | tail -n 5
    %cd {REPO_DIR}
    built_sos = _glob.glob(os.path.join(POINTOPS_SRC, 'pointops_cuda*.so'))
    if built_sos:
        os.makedirs(POINTOPS_CACHE, exist_ok=True)
        for so in built_sos:
            shutil.copy2(so, POINTOPS_CACHE)
        print(f'✓ pointops: built and cached to Drive')
    else:
        print('⚠ pointops: no .so produced — will fall back to JIT at import time')

# --- Compile pointnet2_ops (or restore from cache) ---
# The JIT cache over Drive FUSE is unreliable (always recompiles).
# Instead we explicitly cache the compiled _ext.so and place it in the package dir
# so that `import src.ops.pointnet2_ops._ext` succeeds without JIT.
PNET2_PKG = os.path.join(REPO_DIR, 'src', 'ops', 'pointnet2_ops')
sys.path.insert(0, REPO_DIR)

pnet2_cached = _glob.glob(os.path.join(PNET2_CACHE, '_ext*.so'))
if pnet2_cached:
    for so in pnet2_cached:
        shutil.copy2(so, PNET2_PKG)
    t0 = _time.time()
    from src.ops.pointnet2_ops import pointnet2_utils
    print(f'✓ pointnet2_ops: restored .so from Drive cache ({_time.time()-t0:.1f}s)')
else:
    print('pointnet2_ops: JIT compiling (first time for this env)...')
    t0 = _time.time()
    from src.ops.pointnet2_ops import pointnet2_utils  # triggers JIT
    dt = _time.time() - t0
    # Find the compiled .so in the JIT cache and save to Drive
    _jit_cache = os.path.expanduser('~/.cache/torch_extensions')
    _jit_sos = _glob.glob(os.path.join(_jit_cache, '**', '_ext.so'), recursive=True)
    if _jit_sos:
        os.makedirs(PNET2_CACHE, exist_ok=True)
        shutil.copy2(_jit_sos[0], PNET2_CACHE)
        shutil.copy2(_jit_sos[0], PNET2_PKG)
        print(f'✓ pointnet2_ops: JIT compiled ({dt:.1f}s) — cached for next time')
    else:
        print(f'✓ pointnet2_ops: JIT compiled ({dt:.1f}s) — could not find .so to cache')

print(f'\n✓ All CUDA extensions ready (env: {_cache_tag})')

In [ ]:
# Verify imports
import sys
sys.path.insert(0, REPO_DIR)

from src.generative import get_strategy, STRATEGIES
from src.utils.config import load_config
from src.utils.misc import set_seed

print(f'Available strategies: {list(STRATEGIES.keys())}')
print('All core imports OK!')

## 1.5 Data Preparation

Upload zip files to `/content/` the **first time only** (Colab file browser: folder icon → upload):
- **PU1K**: `/content/pu1k.zip`
- **PUGAN**: `/content/datasets.zip`

Data is extracted to local disk for fast I/O, then cached on Google Drive.  
On subsequent runs, data is restored from Drive — no re-upload needed.

In [ ]:
import os, zipfile, shutil, time as _time

# ============ PATHS ============
DATA_LOCAL = '/content/data'                          # fast local disk (lost on restart)
DATA_DRIVE = '/content/drive/MyDrive/MVA/pointcloud/data'  # persistent cache on Drive
PU1K_ZIP   = '/content/PU1K.zip'
PUGAN_ZIP  = '/content/dataset.zip'
# ===============================

DATASETS = {
    'PU1K': {
        'zip': PU1K_ZIP,
        'h5_rel': 'train/pu1k_poisson_256_poisson_1024_pc_2500_patch50_addpugan.h5',
    },
    'PUGAN': {
        'zip': PUGAN_ZIP,
        'h5_rel': 'train/PUGAN_poisson_256_poisson_1024.h5',
    },
}

os.makedirs(DATA_LOCAL, exist_ok=True)
os.makedirs(DATA_DRIVE, exist_ok=True)

for ds_name, ds_info in DATASETS.items():
    local_dir = os.path.join(DATA_LOCAL, ds_name)
    drive_dir = os.path.join(DATA_DRIVE, ds_name)
    h5_local  = os.path.join(local_dir, ds_info['h5_rel'])
    h5_drive  = os.path.join(drive_dir, ds_info['h5_rel'])

    if os.path.exists(h5_local):
        # Already on local disk (same session)
        print(f'✓ {ds_name}: already on local disk')
        continue

    if os.path.exists(h5_drive):
        # Restore from Drive cache → local (fast copy, no re-extraction)
        t0 = _time.time()
        print(f'{ds_name}: copying from Drive cache to local disk ...')
        shutil.copytree(drive_dir, local_dir, dirs_exist_ok=True)
        print(f'✓ {ds_name}: restored from Drive in {_time.time()-t0:.1f}s')
        continue

    # First time: extract from uploaded zip
    zip_path = ds_info['zip']
    if not os.path.exists(zip_path):
        print(f'⊘ {ds_name}: no zip at {zip_path} and no Drive cache — skipping')
        continue

    print(f'{ds_name}: extracting from {zip_path} ...')
    t0 = _time.time()
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_LOCAL)

    # Auto-fix: if h5 didn't land in expected path, find and move it
    if not os.path.exists(h5_local):
        extracted_h5 = []
        for root, dirs, files in os.walk(DATA_LOCAL):
            for f in files:
                if f.endswith('.h5') and ds_name.lower() in root.lower() + f.lower():
                    extracted_h5.append(os.path.join(root, f))
        if not extracted_h5:
            # Broader search
            for root, dirs, files in os.walk(DATA_LOCAL):
                for f in files:
                    if f.endswith('.h5'):
                        extracted_h5.append(os.path.join(root, f))
        if extracted_h5:
            os.makedirs(os.path.dirname(h5_local), exist_ok=True)
            shutil.move(extracted_h5[0], h5_local)
            print(f'  Moved {os.path.basename(extracted_h5[0])} → {h5_local}')

    if os.path.exists(h5_local):
        print(f'✓ {ds_name}: extracted in {_time.time()-t0:.1f}s')
        # Cache to Drive for next time
        print(f'  Caching to Drive ({drive_dir}) ...')
        shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
        print(f'  Cached to Drive.')
    else:
        print(f'✗ {ds_name}: h5 file not found after extraction. Check zip contents.')
        for root, dirs, files in os.walk(DATA_LOCAL):
            for f in files[:15]:
                print(f'    {os.path.join(root, f)}')

# ---------- Summary ----------
print(f'\nLocal data ({DATA_LOCAL}):')
for root, dirs, files in os.walk(DATA_LOCAL):
    level = root.replace(DATA_LOCAL, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')


## 2. Configuration

Choose your dataset, strategy, and data paths.

In [ ]:
# ============ EDIT THESE ============
DATASET = 'PU1K'                    # 'PU1K' or 'PUGAN'
STRATEGY_NAME = 'ddpm'              # 'ddpm' or 'flow_matching'
# DATA_DIR comes from the extraction cell above (DATA_LOCAL)
DATA_DIR = os.path.join(DATA_LOCAL, DATASET)  # /content/data/PU1K or /content/data/PUGAN
# Checkpoints on Drive so they survive runtime restarts
CHECKPOINT_DIR = os.path.join('/content/drive/MyDrive/MVA/pointcloud', 'checkpoints', DATASET.lower(), STRATEGY_NAME)
OUTPUT_DIR = os.path.join(REPO_DIR, 'outputs', DATASET.lower())
R = 4                               # upsampling rate
GAMMA = 0.5                         # condition interpolation
SEED = 42
# ====================================

from src.utils.config import get_strategy_config

set_seed(SEED)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load config
config_path = os.path.join(REPO_DIR, 'configs', f'{DATASET}.json')
config = load_config(config_path)

# Override data directory in config
dataset_key = 'pu1k_dataset_config' if DATASET == 'PU1K' else 'pugan_dataset_config'
config[dataset_key]['data_dir'] = DATA_DIR

# Get strategy and its config section
strategy = get_strategy(STRATEGY_NAME)
strategy_config = get_strategy_config(config, STRATEGY_NAME)
hyperparams = strategy.compute_hyperparams(**strategy_config)

# Move hyperparams to GPU
for key in hyperparams:
    if key != 'T' and isinstance(hyperparams[key], torch.Tensor):
        hyperparams[key] = hyperparams[key].cuda()

print(f'Dataset: {DATASET}')
print(f'Strategy: {strategy.name}')
print(f'Strategy config: {strategy_config}')
print(f'Data dir: {DATA_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'T = {hyperparams["T"]}')

## 3. Training

Train the model with your chosen strategy. Skip this cell if you already have a checkpoint.

In [ ]:
import time
import torch.nn as nn
from torch.amp import GradScaler, autocast
from src.data.dataset import get_dataloader
from src.models.pointnet2_with_pcld_condition import PointNet2CloudCondition

# Confirm actual GPU right before training (device name can be stale)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print()

# Training params
train_config = config['train_config']
pointnet_config = config['pointnet_config']
trainset_config = config[dataset_key]

N_EPOCHS = 200                      # original paper uses 2000; 200 is enough for comparison
LR = train_config.get('learning_rate', 2e-4)
EPOCHS_PER_CKPT = train_config.get('epochs_per_ckpt', 10)
LOG_EVERY = train_config.get('iters_per_logging', 50)

# DataLoader
trainloader = get_dataloader(trainset_config)
print(f'Training samples: {len(trainloader.dataset)}')
print(f'Batches per epoch: {len(trainloader)}')

# Model
net = PointNet2CloudCondition(pointnet_config).cuda()
optimizer = torch.optim.Adam(net.parameters(), lr=LR)
loss_fn = nn.MSELoss()
scaler = GradScaler('cuda')

# Resume from checkpoint if available
start_epoch = 0
ckpt_files = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pkl')]) if os.path.exists(CHECKPOINT_DIR) else []
if ckpt_files:
    latest = os.path.join(CHECKPOINT_DIR, ckpt_files[-1])
    ckpt = torch.load(latest, map_location='cpu')
    net.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt.get('epoch', 0)
    print(f'Resumed from {latest} at epoch {start_epoch}')

print(f'\nTraining {strategy.name} for {N_EPOCHS} epochs (starting from {start_epoch})...')

In [ ]:
# Training loop (mixed precision)
net.train()
n_iter = start_epoch * len(trainloader)
n_iters_total = N_EPOCHS * len(trainloader)
time0 = time.time()

for epoch in range(start_epoch + 1, N_EPOCHS + 1):
    epoch_loss = 0.0
    for data in trainloader:
        label = data['label'].cuda()
        X = data['complete'].cuda()
        condition = data['partial'].cuda()

        optimizer.zero_grad()
        with autocast('cuda'):
            loss = strategy.training_loss(net, loss_fn, X, hyperparams,
                                         label=label, condition=condition)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        n_iter += 1

        if n_iter % LOG_EVERY == 0:
            print(f'  epoch {epoch}/{N_EPOCHS}  iter {n_iter}/{n_iters_total}  loss: {loss.item():.6f}')

    avg_loss = epoch_loss / len(trainloader)

    if epoch % EPOCHS_PER_CKPT == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'pointnet_ckpt_{epoch}_{avg_loss:.6f}.pkl')
        torch.save({
            'epoch': epoch,
            'iter': n_iter,
            'model_state_dict': net.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'training_time_seconds': int(time.time() - time0),
            'strategy': strategy.name,
        }, ckpt_path)
        print(f'  -> Saved checkpoint: {ckpt_path}')

print(f'\nTraining complete. Total time: {(time.time() - time0)/60:.1f} min')

## 4. Sampling & Evaluation

Generate dense point clouds from the test set and compute metrics.

In [ ]:
from src.scripts.eval import evaluate

# Choose checkpoint
ckpt_files = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pkl')])
if ckpt_files:
    CKPT_PATH = os.path.join(CHECKPOINT_DIR, ckpt_files[-1])
else:
    raise FileNotFoundError(f'No checkpoints in {CHECKPOINT_DIR}')

print(f'Using checkpoint: {CKPT_PATH}')

# Load model
net_eval = PointNet2CloudCondition(pointnet_config).cuda()
ckpt = torch.load(CKPT_PATH, map_location='cpu')
net_eval.load_state_dict(ckpt['model_state_dict'], strict=False)
net_eval.eval()

# Test dataloader
test_config = config[dataset_key].copy()
test_config['batch_size'] = 14
test_config['eval_batch_size'] = 14
testloader = get_dataloader(test_config, phase='test')
print(f'Test samples: {len(testloader.dataset)}')

STEP = 30  # DDIM steps (for DDPM) or Euler steps (for FM)
SAVE_DIR = os.path.join(OUTPUT_DIR, 'samples')
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
with torch.no_grad():
    CD, HD, P2F, meta, metrics = evaluate(
        net=net_eval,
        testloader=testloader,
        strategy=strategy,
        hyperparams=hyperparams,
        print_every_n_steps=strategy_config['T'] // 5,
        scale=config[dataset_key].get('scale', 1),
        compute_cd=True,
        return_all_metrics=True,
        R=R,
        npoints=config[dataset_key]['npoints'],
        T=strategy_config['T'],
        step=STEP,
        save_dir=SAVE_DIR,
        gamma=GAMMA,
    )

print(f'\n{"="*50}')
print(f'Strategy: {strategy.name}')
print(f'Upsampling: {config[dataset_key]["npoints"]} -> {config[dataset_key]["npoints"] * R}')
print(f'Chamfer Distance (CD): {CD:.8f}')
print(f'Hausdorff Distance (HD): {HD:.8f}')
print(f'Point-to-Face (P2F): {P2F:.8f}')
print(f'{"="*50}')

## 5. Single-File Example

Upsample a single `.xyz` point cloud and visualize the result.

In [ ]:
import numpy as np
import open3d
from src.generative.ddpm import DDPMStrategy
from src.utils.pc_utils import pc_normalize, numpy_to_pc

# Path to a single .xyz file
EXAMPLE_FILE = '/content/drive/MyDrive/MVA/pointcloud/data/example/pig.xyz'  # EDIT THIS

if os.path.exists(EXAMPLE_FILE):
    pc = open3d.io.read_point_cloud(EXAMPLE_FILE)
    pts = np.asarray(pc.points, dtype=np.float32)
    pts = pc_normalize(pts)
    condition = torch.from_numpy(pts).unsqueeze(0).cuda()
    npoints = condition.shape[1]
    label = torch.ones(1, dtype=torch.long).cuda() * (R - 1)

    net_eval.eval()
    net_eval.reset_cond_features()

    with torch.no_grad():
        if isinstance(strategy, DDPMStrategy):
            gen, cond_pre, z = strategy.sample_ddim(
                net=net_eval, size=(1, npoints * R, 3),
                hyperparams=hyperparams, label=label,
                condition=condition, R=R, gamma=GAMMA, step=STEP,
            )
        else:
            gen, cond_pre, z = strategy.sample(
                net=net_eval, size=(1, npoints * R, 3),
                hyperparams=hyperparams, label=label,
                condition=condition, R=R, gamma=GAMMA, num_steps=STEP,
            )

    gen_np = gen[0].cpu().numpy()
    print(f'Input: {npoints} points -> Generated: {gen_np.shape[0]} points')

    # Save result
    out_path = os.path.join(OUTPUT_DIR, 'example_output.xyz')
    open3d.io.write_point_cloud(out_path, numpy_to_pc(gen_np))
    print(f'Saved to: {out_path}')
else:
    print(f'Example file not found: {EXAMPLE_FILE}')
    print('Edit EXAMPLE_FILE path above, or skip this cell.')


## 6. Compare Strategies

Train/evaluate both DDPM and Flow Matching on the same data to compare.

In [ ]:
# Quick comparison template (fill in checkpoints)
results = {}

for strat_name in ['ddpm', 'flow_matching']:
    ckpt_path = os.path.join(REPO_DIR, 'outputs', DATASET.lower(), 'checkpoints',
                             f'{strat_name}_final.pkl')  # adjust filename
    if not os.path.exists(ckpt_path):
        print(f'Skipping {strat_name} — no checkpoint at {ckpt_path}')
        continue

    strat = get_strategy(strat_name)
    strat_cfg = get_strategy_config(config, strat_name)
    hp = strat.compute_hyperparams(**strat_cfg)
    for key in hp:
        if key != 'T' and isinstance(hp[key], torch.Tensor):
            hp[key] = hp[key].cuda()

    net_cmp = PointNet2CloudCondition(pointnet_config).cuda()
    ckpt = torch.load(ckpt_path, map_location='cpu')
    net_cmp.load_state_dict(ckpt['model_state_dict'], strict=False)
    net_cmp.eval()

    with torch.no_grad():
        cd, hd, p2f, _, _ = evaluate(
            net=net_cmp, testloader=testloader, strategy=strat,
            hyperparams=hp, R=R, npoints=config[dataset_key]['npoints'],
            T=strat_cfg['T'], step=30, gamma=GAMMA,
            save_dir=os.path.join(OUTPUT_DIR, f'samples_{strat_name}'),
            save_xyz=False,
        )
    results[strat_name] = {'CD': cd, 'HD': hd, 'P2F': p2f}
    print(f'{strat_name}: CD={cd:.8f}  HD={hd:.8f}  P2F={p2f:.8f}')

if results:
    print('\n--- Comparison ---')
    for name, m in results.items():
        print(f"  {name:15s}  CD={m['CD']:.8f}  HD={m['HD']:.8f}  P2F={m['P2F']:.8f}")